In [ ]:
import grewpy
from grewpy import Corpus, CorpusDraft, Request
from collections import Counter
import sys
sys.path.insert(1, "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/tod")

import tod.corpus
import tod.outliers
import tod.clustering
import tod.plotting
import tod.dimension_reduction_classic

treebank_path = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/data/input/Universal_Dependencies/ud-treebanks-v2.15/UD_French-GSD"
# treebank_path = "/Users/madalina/Downloads/bUD_English-GUM"
grew_pattern = "pattern{X[upos<>PUNCT]}"
patterns_text_file = "/Users/madalina/Documents/M2TAL/stage/check_coherent_labels/scripts/3. probability_matrix/patterns_all_nodes.txt"
analysed_category = "all_nodes"

corpus = tod.corpus.Corpus(
    treebank_path=treebank_path,
    grew_pattern=grew_pattern,
    patterns_text_file=patterns_text_file,
    use_sud=False,
    matrix_type="coverage",
    excluded_feature_patterns=[r"CxnElt=", r"Cxn=", r"own", r"Gender"]
        # excluded_feature_patterns=[r"CxnElt=", r"Cxn=", r"XML=", r"PDTB=", r"SplitAnte=", r"MSeg=", r"Entity=", r"Discourse=", r"Bridge=", r"own"]
)

In [27]:
import numpy as np
from scipy.spatial.distance import pdist, squareform

x = np.array([[2, 0, 2], [2, 2, 3], [-2, 4, 5], [0, 1, 9], [2, 2, 4]])
# x = np.array([[2, 2, 4], [-2, 4, 5]])

y = pdist(x)
squareform(y)

array([[0.        , 2.23606798, 6.40312424, 7.34846923, 2.82842712],
       [2.23606798, 0.        , 4.89897949, 6.40312424, 1.        ],
       [6.40312424, 4.89897949, 0.        , 5.38516481, 4.58257569],
       [7.34846923, 6.40312424, 5.38516481, 0.        , 5.47722558],
       [2.82842712, 1.        , 4.58257569, 5.47722558, 0.        ]])

In [15]:
# corpus.lexunit2idx(("pas", "ADV"))
# corpus.lexunit2idx(("divers", "DET"))
# corpus.lexunit2idx(("jamais", "ADV"))
# corpus.lexunit2idx(("have", "AUX"))
# corpus.lexunit2idx(("from", "SCONJ"))
corpus.lexunit2idx(("deux", "NUM"))

# corpus._lexunit2idx

998

In [28]:
x = corpus.feature_matrix
y = squareform(pdist(x))
y

array([[0.        , 0.06420302, 0.42645197, ..., 0.19975641, 0.2089273 ,
        0.05854695],
       [0.06420302, 0.        , 0.41879204, ..., 0.19865229, 0.20430046,
        0.07099934],
       [0.42645197, 0.41879204, 0.        , ..., 0.39426784, 0.40705466,
        0.4232117 ],
       ...,
       [0.19975641, 0.19865229, 0.39426784, ..., 0.        , 0.18277224,
        0.19945653],
       [0.2089273 , 0.20430046, 0.40705466, ..., 0.18277224, 0.        ,
        0.22483907],
       [0.05854695, 0.07099934, 0.4232117 , ..., 0.19945653, 0.22483907,
        0.        ]])

In [6]:
np.argsort(y[1559])

array([1559, 1242,  918, ..., 1054, 1277, 1296])

In [7]:
corpus.idx2lexunit(1970)

('town', 'NOUN')

In [29]:
rows_minimatrix, columns_minimatrix = set(), set()
for f in corpus._feature2idx:
    rows_minimatrix.add(f.split(":")[2])
    columns_minimatrix.add(f.split(":")[-1])
rows_minimatrix = list(rows_minimatrix)
columns_minimatrix = list(columns_minimatrix)
print(rows_minimatrix)
print(columns_minimatrix)

['child', 'prev', 'parent', 'next']
['ExtPos=PRON', 'ExtPos=NUM', 'tense', 'rel_shallow=goeswith', 'rel_shallow=obl', 'rel_shallow=advcl', 'ExtPos=ADJ', 'ExtPos=INTJ', 'rel_shallow=dep', 'Mood=Ind', 'ExtPos=ADV', 'rel_shallow=csubj', 'rel_shallow=flat', 'upos=X', 'upos=SYM', 'upos=AUX', 'outer', 'upos=PROPN', 'rel_shallow=vocative', 'rel_shallow=mark', 'rel_shallow=nummod', 'Voice=Act', 'ExtPos=ADP', 'pv', 'Polarity=Pos', 'name', 'PronType=Exc', 'rel_shallow=discourse', 'upos=DET', 'ExtPos=VERB', 'rel_shallow=ccomp', 'PronType=Int', 'Definite=Ind', 'PronType=Prs', 'Voice=Pass', 'PronType=Dem', 'comp', 'Subject=SubjRaising', 'cleft', 'ExtPos=CCONJ', 'Subject=Instantiated', 'upos=ADV', 'Tense=Imp', 'upos=ADP', 'rel_shallow=orphan', 'rel_shallow=advmod', 'rel_shallow=compound', 'position=after', 'Subject=OblRaising', 'ExtPos=NOUN', 'caus', 'ExtPos=SCONJ', 'upos=PRON', 'position=before', 'Subject=ObjRaising', 'rel_shallow=cc', 'Polarity=Neg', 'Person=2', 'upos=VERB', 'PronType=Rel', 'Perso

In [30]:
mini_matrices = []
top_similar_indices = list(np.argsort(y[998]))[:50]
# print([corpus.idx2lexunit(i) for i in top_similar_indices])
for i in top_similar_indices:
    mini_matrix = np.zeros((len(rows_minimatrix), len(columns_minimatrix)))
    current_word_row = list(x[i])
    for idx, value in enumerate(current_word_row):
        feature = corpus.idx2feature(idx)
        row = rows_minimatrix.index(feature.split(":")[2])
        column = columns_minimatrix.index(feature.split(":")[-1])
        mini_matrix[row, column] = x[i, idx]
    mini_matrices.append(mini_matrix)



In [21]:
len(mini_matrices)

50

In [31]:
diff = mini_matrices[0] - mini_matrices[1]
biggest_diffs = np.unravel_index(np.argsort(abs(diff), axis=None)[-5:], diff.shape)
print(biggest_diffs)
for i in range(5):
    print(rows_minimatrix[biggest_diffs[0][i]], columns_minimatrix[biggest_diffs[1][i]], diff[biggest_diffs[0][i], biggest_diffs[1][i]])

(array([2, 1, 1, 1, 1]), array([ 86, 106,  97,  28,  43]))
parent upos=NOUN -0.009023134267871019
prev PronType=Art 0.01141500715423538
prev Definite=Def 0.01141500715423538
prev upos=DET 0.012659921496878654
prev upos=ADP -0.01449257063305514


In [344]:
diff[1,64]

0.0

In [345]:
diff[biggest_diff]

-0.005571150197865965

In [346]:
print(rows_minimatrix[biggest_diff[0]])
print(columns_minimatrix[biggest_diff[1]])

prev
Person=3


in the end i just differentiate the vectors, not the mini matrices as it really just leads to the same thing

In [32]:
complementarity_scores = {}
for i in top_similar_indices:
    if i == top_similar_indices[0]:
        highest_feats = np.argsort(x[top_similar_indices[0]])[::-1][:5].tolist()
        complementarity_scores[corpus.idx2lexunit(i)] = {"score": 0, "features": [corpus.idx2feature(idx) for idx in highest_feats]}
        continue
    diff = x[top_similar_indices[0]] - x[i] # difference between the element itself and the similar one
    
    complementarity_scores[corpus.idx2lexunit(i)]= {}
    complementarity_scores[corpus.idx2lexunit(i)]["score"] = np.sum(abs(diff))
    highest_diff_feats_idx = np.argsort(np.abs(diff))[::-1][:5].tolist()
    # print(highest_diff_feats_idx)
    complementarity_scores[corpus.idx2lexunit(i)]["features"] = [(corpus.idx2feature(idx), float(diff[idx])) for idx in highest_diff_feats_idx]


complementarity_scores

{('deux', 'NUM'): {'score': 0,
  'features': ['node:X:parent:position=after',
   'node:X:parent:upos=NOUN',
   'node:X:next:upos=NOUN',
   'node:X:prev:upos=DET',
   'node:X:prev:Definite=Def']},
 ('trois', 'NUM'): {'score': 0.21740566612183426,
  'features': [('node:X:prev:upos=ADP', -0.01449257063305514),
   ('node:X:prev:upos=DET', 0.012659921496878654),
   ('node:X:prev:Definite=Def', 0.01141500715423538),
   ('node:X:prev:PronType=Art', 0.01141500715423538),
   ('node:X:parent:upos=NOUN', -0.009023134267871019)]},
 ('cinq', 'NUM'): {'score': 0.2827511036939261,
  'features': [('node:X:next:upos=NOUN', -0.0195566875161928),
   ('node:X:prev:upos=ADP', -0.011001080919033158),
   ('node:X:parent:upos=NOUN', -0.008337872159518561),
   ('node:X:prev:upos=PUNCT', 0.008198961464881116),
   ('node:X:prev:PronType=Art', -0.0074481976175846745)]},
 ('quatre', 'NUM'): {'score': 0.25086737600807857,
  'features': [('node:X:prev:upos=ADP', -0.01828360686365791),
   ('node:X:prev:upos=DET', 0.0

In [33]:
arr = np.array([[1, 2, 3], [4, 5, 6]])
np.sum(arr)

21

In [34]:
import plotly.graph_objects as go
import numpy as np

def plot_complementarity_with_axis(comp_scores: dict, top_k_features: int = 5):
    # --- 1. Data Preparation (Same as before) ---
    items = list(comp_scores.items())
    rest = sorted(items[1:], key=lambda kv: kv[1]['score'])
    ordered = [items[0]] + rest

    def fmt_key(k):
        lemma, upos = k
        return f"<b>{lemma}</b><br>({upos})"

    def fmt_features_hover(feats):
        if not feats: return ""
        if isinstance(feats[0], tuple):
            lines = [f"{f}: {v:+.3f}" for f, v in feats[:top_k_features]]
        else:
            lines = [str(f) for f in feats[:top_k_features]]
        return "<br>".join(lines)
    
    labels = [fmt_key(k) for k, _ in ordered]
    xvals = [0.0] + [v['score'] for _, v in ordered[1:]]
    yvals = np.zeros(len(ordered))
    
    hovertexts = []
    for i, (k, v) in enumerate(ordered):
        feat_str = fmt_features_hover(v['features'])
        score_str = f"{v['score']:.4f}" if i > 0 else "0 (Base)"
        hovertexts.append(f"{fmt_key(k)}<br>Score: {score_str}<br><br>Features:<br>{feat_str}")

    fig = go.Figure()

    # --- 2. Draw the Points ---
    fig.add_trace(go.Scatter(
        x=xvals, y=yvals,
        mode="markers",
        marker=dict(size=14, color='#2c3e50', line=dict(width=2, color='white')), # Added white border for contrast
        hovertext=hovertexts,
        hoverinfo="text",
        showlegend=False,
        name="Words"
    ))

    # --- 3. Draw the Axis Line (The Fix) ---
    # We draw a line shape from slightly before the first point to slightly after the last
    min_x, max_x = min(xvals), max(xvals)
    padding = (max_x - min_x) * 0.05 # 5% padding
    
    fig.add_shape(
        type="line",
        x0=min_x - padding, y0=0,
        x1=max_x + padding, y1=0,
        line=dict(color="black", width=3),
        layer="below" # Ensures the line is BEHIND the dots
    )

    # --- 4. Staggered Annotations ---
    stagger_offsets = [-40, 40, -90, 90, -140, 140]
    
    for i, (xi, lbl) in enumerate(zip(xvals, labels)):
        offset = stagger_offsets[i % len(stagger_offsets)]
        y_anchor = "bottom" if offset < 0 else "top"
        
        # Add a subtle vertical connector line from the axis to the text
        # This helps visually connect floating text back to the axis line
        fig.add_shape(
            type="line",
            x0=xi, y0=0,
            x1=xi, y1=0, # The annotation arrow handles the rest, but you can draw a full line if needed
            line=dict(color="#ddd", width=1, dash="dot"),
            layer="below"
        )

        fig.add_annotation(
            x=xi, y=0,
            text=lbl,
            showarrow=True,
            arrowhead=0,
            arrowwidth=1.5,
            arrowcolor="#555",
            ax=0, 
            ay=offset,
            font=dict(size=11),
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="#ddd",
            borderwidth=1,
            borderpad=4
        )

    # --- 5. Clean Layout ---
    fig.update_layout(
        title="Complementarity Distance",
        xaxis_title="Score",
        yaxis=dict(visible=False, range=[-1, 1]),
        xaxis=dict(
            visible=True, 
            showgrid=False, 
            zeroline=False, # We drew our own line, so turn off the default
            showticklabels=True
        ),
        margin=dict(t=100, b=100, l=50, r=50),
        height=600,
        width=1500,
        plot_bgcolor="white"
    )

    return fig

fig = plot_complementarity_with_axis(complementarity_scores)
fig.show()

In [350]:
fig.write_html("from.html")